# Construcción de la base final

Limpieza reproducible de los datos del MEN sobre deserción escolar en Colombia, 2011 a 2024. Ejecuta las celdas en orden desde la carpeta del proyecto. El CSV original se conserva y la única salida es `limpieza/datos_limpios.csv`.


### Librerías


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


## Limpieza de la base


### Definición de campos


In [ ]:
TASAS = ["desercion", "desercion_transicion", "desercion_primaria",
"desercion_secundaria", "desercion_media", "aprobacion", "reprobacion",
"cobertura_neta", "cobertura_bruta"]
BASE = ["ano", "c_digo_departamento", "departamento", "poblacion_5_16"] + TASAS

### Funciones de limpieza


In [ ]:
def numero(serie):
    # El CSV oficial usa punto decimal. Se admite coma decimal sin separador de miles.
    texto = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="coerce")
    presentes = texto.notna() & ~texto.isin(["", "nan", "None", "null"])
    if (presentes & convertido.isna()).any():
        raise ValueError(f"Valores no numéricos en {serie.name}; revisar antes de continuar")
    return convertido

def preparar(entrada, salida):
    salida.parent.mkdir(parents=True, exist_ok=True)
    original = pd.read_csv(entrada, dtype="string", keep_default_na=True)
    faltan = set(BASE) - set(original.columns)
    if faltan:
        raise ValueError(f"Faltan columnas requeridas: {sorted(faltan)}")
    duplicados_exactos = int(original.duplicated().sum())
    datos = original.drop_duplicates().loc[:, BASE].copy()
    for c in BASE:
        if c != "departamento":
            datos[c] = numero(datos[c])
    for c in ["ano", "c_digo_departamento", "poblacion_5_16"]:
        if datos[c].isna().any() or ((datos[c] % 1) != 0).any():
            raise ValueError(f"Identificador o población inválido en {c}")
    datos["ano"] = datos["ano"].astype("int64")
    datos["poblacion_5_16"] = datos["poblacion_5_16"].astype("int64")
    datos["c_digo_departamento"] = datos["c_digo_departamento"].astype("int64").astype(str).str.zfill(2)
    if datos["departamento"].isna().any():
        raise ValueError("Departamento ausente")
    antes = datos["departamento"].copy()
    datos["departamento"] = datos["departamento"].str.strip().str.replace(r"\s+", " ", regex=True)
    canon = datos.groupby("c_digo_departamento")["departamento"].agg(lambda s: s.mode().iloc[0]).to_dict()
    canon.update({"11": "Bogotá D.C.", "63": "Quindío"})
    datos["departamento"] = datos["c_digo_departamento"].map(canon)
    corregidos = int((antes != datos["departamento"]).sum())
    if datos.duplicated(["c_digo_departamento", "ano"]).any():
        raise ValueError("Más de un registro por departamento y año; no promediar duplicados conflictivos")
    for c in TASAS:
        v = datos[c].dropna()
        if (v < 0).any() or (c not in ["cobertura_neta", "cobertura_bruta"] and (v > 100).any()):
            raise ValueError(f"Tasa fuera de rango en {c}; revisar el dato original")
    if (datos["poblacion_5_16"] < 0).any():
        raise ValueError("Población negativa")
    # La fuente exporta 3.97 para 3.97 %. Se almacena 0.0397 como proporción para los cálculos y el formato porcentual del tablero.
    datos[TASAS] = datos[TASAS] / 100.0
    datos = datos.sort_values(["c_digo_departamento", "ano"]).reset_index(drop=True)
    datos["rango_desercion"] = np.select(
        [datos.desercion.isna(), datos.desercion < .03, datos.desercion <= .06],
        ["Sin dato", "Baja", "Media"], default="Alta")
    grupo = datos.groupby("c_digo_departamento", sort=False)
    previo = grupo["desercion"].shift(1)
    ano_previo = grupo["ano"].shift(1)
    datos["variacion_interanual_pp"] = ((datos.desercion - previo) * 100).where(datos.ano - ano_previo == 1)
    datos.to_csv(salida, index=False, encoding="utf-8-sig", float_format="%.8f")
    return datos


## Ejecución de las funciones

La ruta se busca dentro del proyecto. La ejecución genera únicamente el CSV limpio, sin reportes ni archivos adicionales.


In [ ]:
carpetas = [Path.cwd(), *Path.cwd().parents]
carpeta_proyecto = next(
    (p for p in carpetas if (p / "limpieza" / "datos_originales_men.csv").is_file()),
    None,
)
if carpeta_proyecto is None:
    raise FileNotFoundError(
        "Abre Jupyter desde la carpeta del proyecto. "
        "Debe existir limpieza/datos_originales_men.csv."
    )

entrada = carpeta_proyecto / "limpieza" / "datos_originales_men.csv"
salida = carpeta_proyecto / "limpieza" / "datos_limpios.csv"
datos = preparar(entrada, salida)
print("CSV limpio generado en limpieza/datos_limpios.csv")
